# Track A, Session 1, Feature Engineering and Baselines (your turn)
Fill the `# TODO` lines and the **Observe** cells. Every feature we build comes from an observation made in Part A.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
plt.rcParams.update({"figure.figsize": (11, 3.6), "axes.titleweight": "bold"})
BLUE, ORANGE, GREY = "#1f6feb", "#fb8500", "#9aa0a6"
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data"; PROC = DATA / "processed"; PROC.mkdir(parents=True, exist_ok=True)
IMG = ROOT / "images"; (IMG / "partC").mkdir(parents=True, exist_ok=True); (IMG / "concepts").mkdir(parents=True, exist_ok=True)

## 1. Rebuild the Part B protocol (condensed)
Same steps as Part B, in one cell: parse and sort time, de-duplicate and aggregate weather, merge, clip impossible values, flag and interpolate the target.

In [ ]:
energy = pd.read_csv(DATA / "energy_dataset.csv")
energy["time"] = pd.to_datetime(energy["time"], utc=True)
energy = energy.sort_values("time").reset_index(drop=True)

weather = pd.read_csv(DATA / "weather_features.csv")
weather["time"] = pd.to_datetime(weather["dt_iso"], utc=True)
weather["city_name"] = weather["city_name"].str.strip()
weather = weather.drop_duplicates(subset=["time", "city_name"])
wnum = [c for c in weather.select_dtypes("number").columns if c != "weather_id"]
wagg = weather.groupby("time")[wnum].mean().reset_index()
for c in ["temp", "temp_min", "temp_max"]: wagg[c] -= 273.15

df = energy.merge(wagg, on="time", how="left")
df["pressure"] = df["pressure"].clip(870, 1085)
df["wind_speed"] = df["wind_speed"].clip(0, 40)

TARGET, REFERENCE = "total load actual", "total load forecast"
df["target_missing"] = df[TARGET].isna()
df["target_filled"]  = df.set_index("time")[TARGET].interpolate("time").values
print("df", df.shape, "| target gaps flagged:", int(df["target_missing"].sum()))

## 2. The day-ahead setup
At time t, we predict demand at **t + 24 hours**, using only information available at time t.

Consequence for the feature table: for a target hour T, every feature must be **at least 24 hours old**. So the most recent demand lag we may use is `lag_24` (the same hour yesterday), never `lag_1`. This is the leakage audit of Part B turned into a modeling rule.

A full 24-output forecaster (t+1 ... t+24) is the project extension; today one clean horizon teaches the correct discipline.

## 3. Calendar features
From Part A: daily rhythm, weekly effect, yearly cycle. **Your turn:** build hour, day_of_week, month, is_weekend.

In [ ]:
df["hour"] = df["time"].dt.hour
df["day_of_week"] = ...        # TODO: 0 = Monday (dt.dayofweek)
df["month"] = ...              # TODO
df["is_weekend"] = ...         # TODO: 1 if Saturday or Sunday, else 0
df[["time", "hour", "day_of_week", "month", "is_weekend"]].head(3)

## 4. Cyclic encoding
Hour 23 and hour 0 are one hour apart, but as raw numbers they look 23 apart. **Your turn:** encode the hour on a circle with sin and cos (day of week and month are given).

In [ ]:
df["hour_sin"] = ...   # TODO: np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = ...   # TODO
df["dow_sin"]  = np.sin(2 * np.pi * df["day_of_week"] / 7)
df["dow_cos"]  = np.cos(2 * np.pi * df["day_of_week"] / 7)
df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)

## 5. Lag features
From Part A: autocorrelation 0.70 at t-24 and 0.66 at t-168. **Your turn:** build lags 24, 48 and 168 on `target_filled`. Day-ahead rule: the smallest allowed lag is 24.

In [ ]:
for lag in (24, 48, 168):
    df[f"load_lag_{lag}"] = ...   # TODO: shift target_filled by lag hours
df[["time", "target_filled", "load_lag_24", "load_lag_168"]].iloc[170:173]

## 6. Rolling features (past only)
A rolling mean summarizes the recent level of demand. Day-ahead rule again: the window must end at t-24, so we shift by 24 **before** rolling. A rolling window without the shift would include hours we do not have yet at prediction time.

In [ ]:
df["load_roll_mean_24"]  = df["target_filled"].shift(24).rolling(24).mean()
df["load_roll_std_24"]   = df["target_filled"].shift(24).rolling(24).std()
df["load_roll_mean_168"] = df["target_filled"].shift(24).rolling(168).mean()
print("rolling windows end 24 h before the target hour: safe by construction")

## 7. Weather as a forecast proxy
From Part A: the temperature effect is real and non-linear (cooling above 25 C). At prediction time, the true weather of t+24 is not observed yet; what exists in a real system is a **weather forecast**. In this educational dataset we use the observed weather at the target hour as a proxy for that forecast, and we write the assumption down.

In [ ]:
WEATHER_FEATURES = ["temp", "humidity", "wind_speed"]
print("Assumption (documented): observed weather at the target hour stands in for a day-ahead weather forecast.")
df[WEATHER_FEATURES].describe().loc[["min", "mean", "max"]].round(1)

## 8. Assemble X and y, split by time
**Your turn:** complete the temporal split (train up to 2016, validation 2017, test 2018).

In [ ]:
FEATURES = (["hour", "day_of_week", "month", "is_weekend",
             "hour_sin", "hour_cos", "dow_sin", "dow_cos", "month_sin", "month_cos",
             "load_lag_24", "load_lag_48", "load_lag_168",
             "load_roll_mean_24", "load_roll_std_24", "load_roll_mean_168"]
            + WEATHER_FEATURES)

data = df[~df["target_missing"]].dropna(subset=FEATURES).copy()
year = data["time"].dt.year
train = ...   # TODO: year <= 2016
val   = ...   # TODO: year == 2017
test  = ...   # TODO: year == 2018
for n, d in [("train", train), ("val", val), ("test", test)]:
    print(f"{n:5s} {d.shape[0]:6d}   {d['time'].min().date()} -> {d['time'].max().date()}")

## The three metrics we use
Every forecast, baseline or model, is scored the same way. **MAE**, the mean absolute error, is the average error in megawatts and the easiest to read. **RMSE** squares the errors before averaging, so it punishes the large misses more. **MAPE** is the error as a percentage, which lets us compare with the provided forecast. MAE is our primary metric.

## 9. Baselines
A baseline tells us whether machine learning adds value. **Your turn:** evaluate the three references on the 2018 test year: same hour yesterday (`load_lag_24` alone), same hour last week (`load_lag_168` alone), and the provided `total load forecast`.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

def evaluate(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true, float), np.asarray(y_pred, float)
    return {"MAE": mean_absolute_error(y_true, y_pred),
            "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
            "MAPE_%": float(np.mean(np.abs((y_true - y_pred) / y_true)) * 100)}

ref = test.dropna(subset=[REFERENCE])
baselines = pd.DataFrame([
    {"model": "Naive: same hour yesterday",  **evaluate(test[TARGET], ...)},   # TODO
    {"model": "Naive: same hour last week",  **evaluate(test[TARGET], ...)},   # TODO
    {"model": "Provided forecast (reference)", **evaluate(ref[TARGET], ref[REFERENCE])},
]).set_index("model").round(2)
baselines

In [ ]:
wk = test[(test["time"] >= "2018-03-05") & (test["time"] < "2018-03-12")]
fig, ax = plt.subplots()
ax.plot(wk["time"], wk[TARGET], color=BLUE, label="actual")
ax.plot(wk["time"], wk["load_lag_24"], color=ORANGE, ls="--", label="naive: same hour yesterday")
ax.set_title("What the model must beat (one test week)"); ax.set_ylabel("MW"); ax.legend()
fig.savefig(IMG / "partC" / "baseline_week.png", dpi=130, bbox_inches="tight"); plt.show()

**Observe.** Which baseline is hardest to beat? Where does the daily naive fail on the weekly plot? What error level must the ML model beat?

## 10. Save the feature table for Session 2

In [ ]:
out = data[["time", TARGET, REFERENCE] + FEATURES]
out.to_csv(PROC / "trackA_features.csv", index=False)
print("saved:", PROC / "trackA_features.csv", "|", out.shape)

## Wrap-up
Every feature came from a Part A observation: rhythm to calendar and cyclic, autocorrelation to lags, recent level to rolling, temperature to a documented weather proxy. The day-ahead rule (nothing younger than 24 h) is the leakage audit turned into design. Session 2: train Ridge and a tree model, compare against these baselines, and check with XAI that the model uses the signals we expected.